SECTION 1

In [ ]:
import pandas as pd
import numpy as np

from tqdm import tqdm

SECTION 2
Data Loading

In [ ]:
MAX_ROWS = 2_000_000

In [ ]:
data = []

movie_id = None

rows_loaded = 0

with open(
    "combined_data_1.txt",
    "r"
) as f:

    for line in tqdm(f):

        line = line.strip()

        if line.endswith(":"):

            movie_id = int(
                line[:-1]
            )

        else:

            user_id, rating, date = (
                line.split(",")
            )

            data.append([
                int(user_id),
                movie_id,
                int(rating),
                pd.to_datetime(date)
            ])

            rows_loaded += 1

            if rows_loaded >= MAX_ROWS:
                break

2000360it [12:21, 2698.24it/s]


In [ ]:
ratings_df = pd.DataFrame(
    data,
    columns=[
        "user_id",
        "movie_id",
        "rating",
        "date"
    ]
)

ratings_df.head()

,user_id,movie_id,rating,date
0,1488844,1,3,2005-09-06
1,822109,1,5,2005-05-13
2,885013,1,4,2005-10-19
3,30878,1,4,2005-12-26
4,823519,1,3,2004-05-03


In [ ]:
ratings_df.shape

(2000000, 4)

SECTION 3
Global Temporal Split

In [ ]:
ratings_df = ratings_df.sort_values(
    "date"
)

split_idx = int(
    len(ratings_df) * 0.8
)

train_df = ratings_df.iloc[
    :split_idx
].copy()

test_df = ratings_df.iloc[
    split_idx:
].copy()

SECTION 4
User Coverage

(Check whether test users were seen during training)

In [ ]:
train_users = set(
    train_df["user_id"]
)

test_users = set(
    test_df["user_id"]
)

seen_users = (
    test_users &
    train_users
)

unseen_users = (
    test_users -
    train_users
)

print(
    "Train Users:",
    len(train_users)
)

print(
    "Test Users:",
    len(test_users)
)

print(
    "Seen Test Users:",
    len(seen_users)
)

print(
    "Unseen Test Users:",
    len(unseen_users)
)

print(
    "Cold Start Users (%):",
    round(
        100 *
        len(unseen_users)
        /
        len(test_users),
        2
    )
)

Train Users: 285109
Test Users: 148918
Seen Test Users: 91582
Unseen Test Users: 57336
Cold Start Users (%): 38.5


SECTION 5
Movie Coverage

(Check whether test movies were seen during training)

In [ ]:
train_movies = set(
    train_df["movie_id"]
)

test_movies = set(
    test_df["movie_id"]
)

unseen_movies = (
    test_movies -
    train_movies
)

print(
    "Train Movies:",
    len(train_movies)
)

print(
    "Test Movies:",
    len(test_movies)
)

print(
    "Unseen Movies:",
    len(unseen_movies)
)

print(
    "Cold Start Movies (%):",
    round(
        100 *
        len(unseen_movies)
        /
        len(test_movies),
        2
    )
)

Train Movies: 356
Test Movies: 361
Unseen Movies: 5
Cold Start Movies (%): 1.39


SECTION 6
Interaction Coverage

(How many test ratings involve cold-start entities?)

In [ ]:
cold_user_rows = test_df[
    ~test_df["user_id"].isin(
        train_users
    )
]

cold_movie_rows = test_df[
    ~test_df["movie_id"].isin(
        train_movies
    )
]

print(
    "Ratings with Unseen Users:",
    len(cold_user_rows)
)

print(
    "Ratings with Unseen Movies:",
    len(cold_movie_rows)
)

print(
    "Cold User Rating %:",
    round(
        100 *
        len(cold_user_rows)
        /
        len(test_df),
        2
    )
)

print(
    "Cold Movie Rating %:",
    round(
        100 *
        len(cold_movie_rows)
        /
        len(test_df),
        2
    )
)

Ratings with Unseen Users: 192987
Ratings with Unseen Movies: 2309
Cold User Rating %: 48.25
Cold Movie Rating %: 0.58
